# 01 · EDA: the application table

**Goal:** understand the main table, check the known traps, and collect the numbers for the
*Data* section of the report. Run all cells from top to bottom (takes about 1–2 minutes).

At the end you write two things yourself: **Findings** (what you saw, with numbers) and
**Decisions** (cleaning rules for the feature code). The `facts` cell collects the key numbers for you.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from creditrisk.config import get_path, set_seed

SEED = set_seed()
INTERIM, FIG = get_path("interim"), get_path("figures")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
facts = {}  # key numbers for the Findings, printed at the end


def save(fig, name):
    """Save a figure for the report."""
    fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
    print(f"saved reports/figures/{name}")


app = pd.read_parquet(INTERIM / "application_train.parquet")
test = pd.read_parquet(INTERIM / "application_test.parquet")
print("train:", app.shape, "| test:", test.shape)
print(f"memory: {app.memory_usage(deep=True).sum() / 1e6:.0f} MB")

## A. Target

In [ ]:
counts = app["TARGET"].value_counts().sort_index()
rate = app["TARGET"].mean()
facts["n_rows"] = len(app)
facts["n_defaults"] = int(counts[1])
facts["default_rate_%"] = round(100 * rate, 2)
facts["accuracy_if_always_no_default_%"] = round(100 * (1 - rate), 2)
print(counts.to_string())
print(f"default rate: {rate:.2%}")
print(f"always predicting 'no default' already gives {1 - rate:.1%} accuracy")
print("-> accuracy is useless here; use ROC-AUC and PR-AUC instead")

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.bar(["Repaid (0)", "Default (1)"], counts.values, color=["#4C9F70", "#D1495B"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}\n({v / counts.sum():.1%})", ha="center", va="bottom")
ax.set_ylim(0, counts.max() * 1.2)
ax.set(title="Target balance (application_train)", ylabel="Applications")
save(fig, "fig01_target_balance.png")

## B. Missing values

In [ ]:
missing = app.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
facts["cols_with_missing"] = int(len(missing))
facts["cols_over_50pct_missing"] = int((missing > 0.5).sum())
print(f"{len(missing)} of {app.shape[1]} columns have missing values; "
      f"{(missing > 0.5).sum()} are more than 50% missing")
display((100 * missing.head(15)).round(1).rename("missing %").to_frame())

top = missing.head(40)
fig, ax = plt.subplots(figsize=(7, max(3, 0.25 * len(top))))
ax.barh(top.index[::-1], 100 * top.values[::-1], color="#5B8DB8")
ax.axvline(50, color="grey", ls="--", lw=1)
ax.set(title="Columns with the most missing values (top 40)", xlabel="% missing")
save(fig, "fig02_missing.png")

Does *being missing* tell us something about default? If the two rates differ, keep a missing-flag feature.

In [ ]:
rows = []
for col in ["EXT_SOURCE_1", "EXT_SOURCE_3", "OWN_CAR_AGE", "OCCUPATION_TYPE"]:
    is_missing = app[col].isna()
    rows.append({
        "column": col,
        "% missing": round(100 * is_missing.mean(), 1),
        "default % when missing": round(100 * app.loc[is_missing, "TARGET"].mean(), 2),
        "default % when present": round(100 * app.loc[~is_missing, "TARGET"].mean(), 2),
    })
missing_vs_target = pd.DataFrame(rows)
display(missing_vs_target)

## C. Known traps

In [ ]:
anom = app["DAYS_EMPLOYED"] == 365243
facts["days_employed_365243_rows"] = int(anom.sum())
facts["days_employed_365243_share_%"] = round(100 * anom.mean(), 1)
facts["default_%_in_365243_group"] = round(100 * app.loc[anom, "TARGET"].mean(), 2)
facts["default_%_everyone_else"] = round(100 * app.loc[~anom, "TARGET"].mean(), 2)
print(f"DAYS_EMPLOYED == 365243: {anom.sum():,} rows ({anom.mean():.1%})")
print(f"default rate: {app.loc[anom, 'TARGET'].mean():.2%} in this group "
      f"vs {app.loc[~anom, 'TARGET'].mean():.2%} for everyone else")
print("\nNAME_INCOME_TYPE inside this group:")
print(app.loc[anom, "NAME_INCOME_TYPE"].value_counts().loc[lambda s: s > 0].to_string())
xna_org = (app.loc[anom, "ORGANIZATION_TYPE"] == "XNA").mean()
facts["share_of_365243_group_with_org_XNA_%"] = round(100 * xna_org, 1)
print(f"\nShare of this group with ORGANIZATION_TYPE == 'XNA': {xna_org:.1%}")

In [ ]:
xna = int((app["CODE_GENDER"] == "XNA").sum())
facts["gender_XNA_rows"] = xna
print(f"CODE_GENDER == 'XNA': {xna} rows")

income = app["AMT_INCOME_TOTAL"]
facts["income_median"] = float(income.median())
facts["income_max"] = float(income.max())
print("\nAMT_INCOME_TOTAL percentiles:")
print(income.quantile([0.01, 0.25, 0.5, 0.75, 0.99, 1.0]).map("{:,.0f}".format).to_string())

fig, ax = plt.subplots(figsize=(6, 3.6))
sns.histplot(np.log10(income), bins=60, ax=ax, color="#5B8DB8")
ax.set(title="Income is very skewed (log scale)",
       xlabel="log10(AMT_INCOME_TOTAL)", ylabel="Applicants")
save(fig, "fig03_income_log.png")

Near-constant columns (one value in more than 99% of rows) carry almost no information.

In [ ]:
top_share = app.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])
near_constant = top_share[top_share > 0.99].sort_values(ascending=False)
facts["near_constant_columns"] = list(near_constant.index)
print(f"{len(near_constant)} near-constant columns:")
display((100 * near_constant).round(3).rename("% rows with the top value").to_frame())

## D. What relates to default

In [ ]:
num_cols = app.select_dtypes("number").columns.drop(["SK_ID_CURR", "TARGET"])
corr = app[num_cols].corrwith(app["TARGET"], method="spearman").dropna().sort_values()
show = pd.concat([corr.head(15), corr.tail(15)]).drop_duplicates()
facts["most_negative_corr"] = corr.head(5).round(3).to_dict()
facts["most_positive_corr"] = corr.tail(5).round(3).to_dict()

fig, ax = plt.subplots(figsize=(7, max(3, 0.28 * len(show))))
ax.barh(show.index, show.values, color=np.where(show.values > 0, "#D1495B", "#4C9F70"))
ax.axvline(0, color="black", lw=0.8)
ax.set(title="Spearman correlation with TARGET\n(most negative and most positive)",
       xlabel="correlation")
save(fig, "fig04_target_correlations.png")
print("Careful: DAYS_* columns are negative numbers. A positive correlation for DAYS_BIRTH "
      "means YOUNGER people default more.")

In [ ]:
sample = app.sample(n=min(60_000, len(app)), random_state=SEED)
sample = sample.assign(AGE_YEARS=-sample["DAYS_BIRTH"] / 365.25)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"], strict=True):
    sns.kdeplot(data=sample, x=col, hue="TARGET", common_norm=False, fill=True, alpha=0.3, ax=ax)
    ax.set(title=col, xlabel="score (0 to 1)", ylabel="density")
fig.suptitle("External scores by target (0 = repaid, 1 = default)", y=1.04)
save(fig, "fig05_ext_sources.png")

fig, ax = plt.subplots(figsize=(6, 3.6))
sns.kdeplot(data=sample, x="AGE_YEARS", hue="TARGET", common_norm=False,
            fill=True, alpha=0.3, ax=ax)
ax.set(title="Age by target (0 = repaid, 1 = default)", xlabel="age (years)", ylabel="density")
save(fig, "fig06_age.png")

Default rate per category. The red dashed line is the overall default rate and `n` is the group size. Groups with fewer than 100 rows are hidden (their rates are unreliable) and listed above the plot.

In [ ]:
def default_rate_by(col, top=15, min_count=100):
    g = app.groupby(col, observed=True)["TARGET"].agg(default_rate="mean", count="size")
    small = g[g["count"] < min_count]
    if len(small):
        hidden = {str(k): int(v) for k, v in small["count"].items()}
        print(f"{col}: hiding tiny groups (n < {min_count}): {hidden}")
    g = g[g["count"] >= min_count].sort_values("count", ascending=False).head(top)
    return g.sort_values("default_rate")


cat_cols = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
            "NAME_FAMILY_STATUS", "OCCUPATION_TYPE", "ORGANIZATION_TYPE"]
fig, axes = plt.subplots(4, 2, figsize=(14, 17))
for ax, col in zip(axes.flat, cat_cols, strict=False):  # 8 boxes, 7 plots
    g = default_rate_by(col)
    ax.barh(g.index.astype(str), 100 * g["default_rate"], color="#5B8DB8")
    for y, (r, cnt) in enumerate(zip(g["default_rate"], g["count"], strict=True)):
        ax.text(100 * r, y, f"  n={cnt:,}", va="center", fontsize=8)
    ax.axvline(100 * app["TARGET"].mean(), color="#D1495B", ls="--", lw=1)
    ax.set(title=col, xlabel="default rate (%)")
axes.flat[-1].axis("off")
fig.tight_layout()
save(fig, "fig07_default_by_category.png")

## E. Fairness preview
The gap between groups here is the starting point of the fairness section later.

In [ ]:
age_band = pd.cut(-app["DAYS_BIRTH"] / 365.25, bins=[0, 30, 45, 60, 120],
                  labels=["<30", "30-45", "45-60", "60+"], right=False)
by_gender = (
    app[app["CODE_GENDER"] != "XNA"]
    .groupby("CODE_GENDER", observed=True)["TARGET"]
    .agg(default_rate="mean", count="size")
)
by_age = app.groupby(age_band, observed=True)["TARGET"].agg(default_rate="mean", count="size")
facts["default_%_by_gender"] = (100 * by_gender["default_rate"]).round(2).to_dict()
facts["default_%_by_age_band"] = (100 * by_age["default_rate"]).round(2).to_dict()
display(by_gender, by_age)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, (title, g) in zip(axes, [("Gender", by_gender), ("Age band", by_age)], strict=True):
    ax.bar(g.index.astype(str), 100 * g["default_rate"], color="#5B8DB8")
    ax.axhline(100 * app["TARGET"].mean(), color="#D1495B", ls="--", lw=1)
    ax.set(title=f"Default rate by {title.lower()}", xlabel=title, ylabel="default rate (%)")
save(fig, "fig08_default_by_gender_age.png")

## F. Features that copy each other
Pairs with |correlation| > 0.9 (computed on a 60k-row sample to keep it fast). This list is the reason to use PCA on the building columns.

In [ ]:
c = sample[num_cols].corr().abs()
upper = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1))
pairs = upper.stack().loc[lambda s: s > 0.9].sort_values(ascending=False)
facts["pairs_with_abs_corr_over_0.9"] = int(len(pairs))
print(f"{len(pairs)} pairs of numeric columns with |correlation| > 0.9")
display(pairs.rename("abs corr").round(3).to_frame().head(30))

## G. Train vs test
If the test file looks different from train, that is an early hint for the drift section.

In [ ]:
def shares(s):
    v = s.value_counts(normalize=True)
    v.index = v.index.astype(str)
    return 100 * v


contract_share = pd.DataFrame({"train %": shares(app["NAME_CONTRACT_TYPE"]),
                               "test %": shares(test["NAME_CONTRACT_TYPE"])}).round(2)
facts["contract_type_%_train_vs_test"] = contract_share.to_dict()
display(contract_share)

cols = ["EXT_SOURCE_2", "EXT_SOURCE_3", "AMT_CREDIT"]
display(pd.DataFrame({"train mean": app[cols].mean(), "test mean": test[cols].mean()}).round(3))

both = pd.concat([app[cols].assign(set="train"), test[cols].assign(set="test")], ignore_index=True)
both["log10(AMT_CREDIT)"] = np.log10(both["AMT_CREDIT"])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, ["EXT_SOURCE_2", "EXT_SOURCE_3", "log10(AMT_CREDIT)"], strict=True):
    sns.kdeplot(data=both, x=col, hue="set", common_norm=False, ax=ax)
    ax.set(title=col, ylabel="density")
fig.suptitle("Train vs test distributions", y=1.04)
save(fig, "fig09_train_vs_test.png")

## Key facts
Copy these numbers into your Findings below.

In [ ]:
for key, value in facts.items():
    print(f"{key:<40} {value}")

## Findings  ✍️ (write 10–15 bullets, each with a number)

- Default rate is __% (__ defaults out of __). Always predicting "no default" gives __% accuracy, so we use ROC-AUC / PR-AUC, not accuracy.
- __ of 122 columns have missing values; __ are more than 50% missing (mostly building information).
- Missing `EXT_SOURCE_1` / `OCCUPATION_TYPE` is linked to a different default rate (__% vs __%), so missing flags are useful.
- __ rows (__%) have `DAYS_EMPLOYED = 365243`; they are mostly __; their default rate is __% vs __%.
- The strongest single features are __ (Spearman __).
- Younger applicants default more: __% for under-30 vs __% for 60+.
- Default rate: men __%, women __%.
- __ pairs of numeric columns have |corr| > 0.9 (building _AVG/_MODE/_MEDI, AMT_CREDIT vs AMT_GOODS_PRICE, ...).
- Train vs test: ...

## Decisions  ✍️ (cleaning rules for the feature code)

- `DAYS_EMPLOYED == 365243` → NaN, plus a flag column `DAYS_EMPLOYED_ANOM`.
- Drop the __ rows with `CODE_GENDER == "XNA"`.
- Drop near-constant columns: __.
- Building columns: compress with PCA inside the model pipeline (fit on training folds only).
- Keep NaN for tree models; add missing flags + imputation for linear models, KNN, SVM and the MLP.
- Model `AMT_INCOME_TOTAL` on a log scale (or cap at the 99.9th percentile of the training data).